# ArSL Word — Live Webcam Testing

Real-time test for your trained Arabic word model (default: **131-class graduation** `arsl_custom_*`).

### How it works:

1. **Capture** at 640×480 — MediaPipe Holistic on the small frame (fast, stable)
2. **Sliding window** — buffers the last **48 frames** (auto-matched to model input)
3. **Prediction** — BiLSTM + MHA every ~0.55s with stability gating
4. **Sentence building** — confirmed words appended in English + Arabic

### Controls:

| Key | Action |
|-----|--------|
| `Q` | Quit |
| `R` / `C` | Reset sentence |
| `SPACE` | Add space |
| `BACKSPACE` / `D` | Delete last word |

### Default artifacts (graduation):

- `arsl_custom_best.h5`
- `arsl_custom_scaler.npz`
- `arsl_custom_classes.csv`

Set `USE_FILE_PICKER = True` in Cell 2 to browse other checkpoints.

In [11]:
# ===============================
# CELL 1: IMPORTS & SETUP
# ===============================

import cv2
import json
import time
import numpy as np
import pandas as pd
import mediapipe as mp
import tensorflow as tf
from pathlib import Path
from collections import deque

# Arabic text rendering (fixes ??? in window)
from PIL import Image, ImageDraw, ImageFont
import arabic_reshaper
from bidi.algorithm import get_display

print(f'TensorFlow: {tf.__version__}')
print(f'OpenCV: {cv2.__version__}')
print(f'MediaPipe: {mp.__version__}')

# Check GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU detected: {gpus[0].name}')
else:
    print('No GPU — running on CPU')

# --- Arabic-safe text drawing helpers ---
# Batch approach: accumulate text ops then do ONE frame conversion per frame.
_ARABIC_FONT_PATH = 'C:/Windows/Fonts/tahoma.ttf'
_arabic_font_cache = {}

def _get_font(font_size):
    if font_size not in _arabic_font_cache:
        try:
            _arabic_font_cache[font_size] = ImageFont.truetype(_ARABIC_FONT_PATH, font_size)
        except Exception:
            _arabic_font_cache[font_size] = ImageFont.load_default()
    return _arabic_font_cache[font_size]

def frame_to_pil(frame):
    """BGR OpenCV frame → PIL Image (one-time cost per frame)."""
    return Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

def pil_to_frame(pil_img):
    """PIL Image → BGR OpenCV frame."""
    return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

def draw_text_pil(pil_draw, text, pos, font_size=26, color=(255, 255, 255)):
    """Draw Arabic/Latin text on an open PIL.ImageDraw context. color is BGR."""
    try:
        bidi_text = get_display(arabic_reshaper.reshape(text))
    except Exception:
        bidi_text = text
    pil_draw.text(pos, bidi_text, font=_get_font(font_size),
                  fill=(color[2], color[1], color[0]))

def apply_pil_texts(frame, pil_texts):
    """Draw all queued (text, pos, font_size, color) items in ONE round-trip."""
    if not pil_texts:
        return frame
    pil_img  = frame_to_pil(frame)
    pil_draw = ImageDraw.Draw(pil_img)
    for text, pos, fs, clr in pil_texts:
        draw_text_pil(pil_draw, text, pos, fs, clr)
    return pil_to_frame(pil_img)


TensorFlow: 2.10.0
OpenCV: 4.11.0
MediaPipe: 0.10.9
GPU detected: /physical_device:GPU:0


In [12]:
# ===============================
# CELL 2: CONFIGURATION
# ===============================
import tkinter as tk
from tkinter import filedialog, ttk
from pathlib import Path
import json

DIR = Path(r'M:/Term 10/Grad/SLR Main/Words/ArSL Word (Arabic)')
DEFAULT_MODEL   = DIR / 'arsl_custom_best.h5'
DEFAULT_CLASSES = DIR / 'arsl_custom_classes.csv'
DEFAULT_SCALER  = DIR / 'arsl_custom_scaler.npz'

USE_FILE_PICKER = False   # True = popup; False = use graduation defaults above
CONFIG_FILE = str(DIR / 'live_test_config_cache.json')

def get_live_test_config():
    config = {}
    cache = {}
    if Path(CONFIG_FILE).exists():
        try:
            with open(CONFIG_FILE, 'r') as f:
                cache = json.load(f)
        except Exception as e:
            print(f'Warning: Could not load config cache: {e}')

    if not USE_FILE_PICKER:
        config['MODEL_PATH'] = Path(cache.get('MODEL_PATH') or DEFAULT_MODEL)
        config['CLASSES_CSV'] = Path(cache.get('CLASSES_CSV') or DEFAULT_CLASSES)
        config['SCALER_PATH'] = Path(cache.get('SCALER_PATH') or DEFAULT_SCALER)
        return config

    root = tk.Tk()
    root.title('Live Test Configuration')
    root.geometry('620x300')
    root.eval('tk::PlaceWindow . center')

    var_model = tk.StringVar(value=cache.get('MODEL_PATH', str(DEFAULT_MODEL)))
    var_classes = tk.StringVar(value=cache.get('CLASSES_CSV', str(DEFAULT_CLASSES)))
    var_scaler = tk.StringVar(value=cache.get('SCALER_PATH', str(DEFAULT_SCALER)))

    def browse_h5(var):
        f = filedialog.askopenfilename(filetypes=[('H5 Model', '*.h5')])
        if f:
            var.set(f)

    def browse_csv(var):
        f = filedialog.askopenfilename(filetypes=[('CSV Files', '*.csv')])
        if f:
            var.set(f)

    def browse_npz(var):
        f = filedialog.askopenfilename(filetypes=[('NPZ', '*.npz')])
        if f:
            var.set(f)

    ttk.Label(root, text='Model (.h5):').grid(row=0, column=0, sticky='w', padx=10, pady=10)
    ttk.Entry(root, textvariable=var_model, width=42).grid(row=0, column=1, padx=10)
    ttk.Button(root, text='Browse', command=lambda: browse_h5(var_model)).grid(row=0, column=2)

    ttk.Label(root, text='Classes CSV:').grid(row=1, column=0, sticky='w', padx=10, pady=10)
    ttk.Entry(root, textvariable=var_classes, width=42).grid(row=1, column=1, padx=10)
    ttk.Button(root, text='Browse', command=lambda: browse_csv(var_classes)).grid(row=1, column=2)

    ttk.Label(root, text='Scaler (.npz):').grid(row=2, column=0, sticky='w', padx=10, pady=10)
    ttk.Entry(root, textvariable=var_scaler, width=42).grid(row=2, column=1, padx=10)
    ttk.Button(root, text='Browse', command=lambda: browse_npz(var_scaler)).grid(row=2, column=2)

    def start():
        config['MODEL_PATH'] = Path(var_model.get()) if var_model.get() else None
        config['CLASSES_CSV'] = Path(var_classes.get()) if var_classes.get() else None
        config['SCALER_PATH'] = Path(var_scaler.get()) if var_scaler.get() else None
        try:
            with open(CONFIG_FILE, 'w') as f:
                json.dump({
                    'MODEL_PATH': var_model.get(),
                    'CLASSES_CSV': var_classes.get(),
                    'SCALER_PATH': var_scaler.get(),
                }, f, indent=4)
        except Exception as e:
            print(f'Warning: Could not save config cache: {e}')
        root.destroy()

    ttk.Button(root, text='SAVE CONFIG & CONTINUE', command=start).grid(row=3, column=0, columnspan=3, pady=24)
    root.mainloop()
    return config

print('Loading config...')
C = get_live_test_config()
MODEL_PATH = C['MODEL_PATH']
CLASSES_CSV = C['CLASSES_CSV']
SCALER_PATH = C.get('SCALER_PATH', DEFAULT_SCALER)

for p, name in [(MODEL_PATH, 'model'), (CLASSES_CSV, 'classes'), (SCALER_PATH, 'scaler')]:
    if not p or not Path(p).exists():
        raise FileNotFoundError(f'Missing {name}: {p}')

# SEQUENCE_LENGTH set from model.input_shape in Cell 3
SEQUENCE_LENGTH = None

CONFIDENCE_THRESHOLD = 0.42
PREDICTION_INTERVAL  = 0.40   # faster predictions = smoother label updates
STABILITY_WINDOW     = 3      # 3 consecutive → confirmed (was 4)
COOLDOWN_TIME        = 1.5    # seconds before next word accepted (was 1.8)
MIN_ACTIVE_FRAMES    = 0.35

CAMERA_INDEX = 0
CAMERA_WIDTH, CAMERA_HEIGHT = 640, 480
DISPLAY_WIDTH, DISPLAY_HEIGHT = 960, 540   # lighter than 1280×720 — avoids kernel OOM
DRAW_LANDMARKS = True

print(f'Model   : {MODEL_PATH}')
print(f'Classes : {CLASSES_CSV}')
print(f'Scaler  : {SCALER_PATH}')
print(f'Display : {DISPLAY_WIDTH}×{DISPLAY_HEIGHT} (capture {CAMERA_WIDTH}×{CAMERA_HEIGHT})')


Loading config...
Model   : M:\Term 10\Grad\SLR Main\Words\ArSL Word (Arabic)\arsl_custom_best.h5
Classes : M:\Term 10\Grad\SLR Main\Words\ArSL Word (Arabic)\arsl_custom_classes.csv
Scaler  : M:\Term 10\Grad\SLR Main\Words\ArSL Word (Arabic)\arsl_custom_scaler.npz
Display : 960×540 (capture 640×480)


In [13]:
# ===============================
# CELL 3: LOAD MODEL, VOCABULARY & SCALER
# ===============================

class TemporalAttention(tf.keras.layers.Layer):
    """Legacy layer for older ArSL checkpoints only."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name='att_weight', shape=(input_shape[-1], 1),
                                 initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(name='att_bias', shape=(input_shape[1], 1),
                                 initializer='zeros', trainable=True)

    def call(self, x):
        e = tf.nn.tanh(tf.matmul(x, self.W) + self.b)
        a = tf.nn.softmax(e, axis=1)
        return tf.reduce_sum(x * a, axis=1)

print('Loading model...')
try:
    model = tf.keras.models.load_model(str(MODEL_PATH), compile=False)
except Exception:
    model = tf.keras.models.load_model(
        str(MODEL_PATH), compile=False,
        custom_objects={'TemporalAttention': TemporalAttention},
    )
print(f'Model loaded: {model.name} — {model.count_params():,} parameters')

model_input_shape = model.input_shape
SEQUENCE_LENGTH = int(model_input_shape[1])
NUM_FEATURES = int(model_input_shape[2])
NUM_HANDS = 2 if NUM_FEATURES in (126, 258) else 1
LANDMARKS_PER_HAND = 63
seq_batch = np.zeros((1, SEQUENCE_LENGTH, NUM_FEATURES), dtype=np.float32)

print(f'Input shape: {model_input_shape}  →  seq={SEQUENCE_LENGTH}, feat={NUM_FEATURES}')

scaler_path = Path(SCALER_PATH)
if not scaler_path.exists():
    for alt in (Path(MODEL_PATH).parent / 'arsl_custom_scaler.npz',
                Path(MODEL_PATH).parent / 'arsl_scaler_stats.npz'):
        if alt.exists():
            scaler_path = alt
            break

if scaler_path.exists():
    z = np.load(str(scaler_path))
    scaler_mean = z['mean'].astype(np.float32)
    scaler_scale = z['scale'].astype(np.float32)
    print(f'Scaler: {scaler_path.name} ({scaler_mean.shape[0]}-dim)')
else:
    raise FileNotFoundError(f'No scaler found — expected {SCALER_PATH}')

class_df = pd.read_csv(CLASSES_CSV, encoding='utf-8-sig')
index_to_english, index_to_arabic = {}, {}
for _, row in class_df.iterrows():
    idx = int(row['model_class_index'])
    if 'english' in class_df.columns:
        index_to_english[idx] = str(row['english'])
        index_to_arabic[idx] = str(row.get('arabic', row['english']))
    else:
        label = str(row['label_name'])
        index_to_english[idx] = label
        index_to_arabic[idx] = label

num_classes = len(index_to_english)
print(f'{num_classes} classes loaded (model output: {model.output_shape[-1]})')
print('Sample:', [index_to_english[i] for i in range(min(5, num_classes))])


Loading model...
Model loaded: ArSL_100Words_GradProject — 1,642,819 parameters
Input shape: (None, 48, 258)  →  seq=48, feat=258
Scaler: arsl_custom_scaler.npz (258-dim)
131 classes loaded (model output: 131)
Sample: ['0', '1', '2', '3', '4']


In [14]:
# ===============================
# CELL 4: MEDIAPIPE DETECTOR
# ===============================
# Supports 63 features (1 hand), 126 features (2 hands), and 258 features (Holistic)

mp_hands = mp.solutions.hands
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

# Initialize appropriate detector based on features
if NUM_FEATURES == 258:
    detector = mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=0,  # 0 = fastest, minimal accuracy loss for pose
        enable_segmentation=False,
        refine_face_landmarks=False,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )
    print("MediaPipe Holistic detector ready (258 features mode, complexity=0)")
else:
    detector = mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=NUM_HANDS,
        min_detection_confidence=0.6,
        min_tracking_confidence=0.6
    )
    print(f'MediaPipe hand detector ready ({NUM_HANDS} hand(s) mode)')

def extract_landmarks(frame):
    """Extract landmarks dynamically based on NUM_FEATURES."""
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    draw_landmarks = []
    
    if NUM_FEATURES == 258:
        results = detector.process(rgb)
        
        # 1. Pose: 33 x 4 = 132 features
        if results.pose_landmarks:
            pose = np.array([[lm.x, lm.y, lm.z, lm.visibility] for lm in results.pose_landmarks.landmark], dtype=np.float32).flatten()
            draw_landmarks.append(('pose', results.pose_landmarks))
        else:
            pose = np.zeros(132, dtype=np.float32)

        # 2. Left hand: 21 x 3 = 63 features
        if results.left_hand_landmarks:
            lh = np.array([[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark], dtype=np.float32).flatten()
            draw_landmarks.append(('hand', results.left_hand_landmarks))
        else:
            lh = np.zeros(63, dtype=np.float32)

        # 3. Right hand: 21 x 3 = 63 features
        if results.right_hand_landmarks:
            rh = np.array([[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark], dtype=np.float32).flatten()
            draw_landmarks.append(('hand', results.right_hand_landmarks))
        else:
            rh = np.zeros(63, dtype=np.float32)

        combined = np.concatenate([pose, lh, rh])
        return combined, draw_landmarks

    elif NUM_HANDS == 1: # 63 features
        results = detector.process(rgb)
        if results.multi_hand_landmarks:
            lm = results.multi_hand_landmarks[0]
            vec = np.array([[p.x, p.y, p.z] for p in lm.landmark], dtype=np.float32).flatten()
            return vec, [('hand', lm)]
        return np.zeros(NUM_FEATURES, dtype=np.float32), []

    else: # 126 features
        results = detector.process(rgb)
        left_vec = np.zeros(LANDMARKS_PER_HAND, dtype=np.float32)
        right_vec = np.zeros(LANDMARKS_PER_HAND, dtype=np.float32)

        if results.multi_hand_landmarks and results.multi_handedness:
            for hand_lm, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                draw_landmarks.append(('hand', hand_lm))
                label = handedness.classification[0].label
                vec = np.array([[p.x, p.y, p.z] for p in hand_lm.landmark], dtype=np.float32).flatten()

                if label == 'Left':
                    left_vec = vec
                else:
                    right_vec = vec

        combined = np.concatenate([left_vec, right_vec])
        return combined, draw_landmarks


MediaPipe Holistic detector ready (258 features mode, complexity=0)


In [ ]:
# ===============================
# CELL 5: LIVE WEBCAM TESTING  (split view: camera | all-class bars)
# ===============================

FONT = cv2.FONT_HERSHEY_SIMPLEX

# Layout constants
L_W, L_H  = 640, 720        # left panel  (camera + HUD)
R_W, R_H  = 640, 720        # right panel (probability bars)
WIN_W     = L_W + R_W       # 1280 combined window width
WIN_H     = L_H             # 720

TOP_H_L   = 90              # left: top HUD height
BOT_H_L   = 100             # left: bottom sentence bar height
VID_H_L   = L_H - TOP_H_L - BOT_H_L   # 530: video area height

HDR_H_R   = 42              # right: header height
BAR_ROW_H = 26              # right: height per prediction row
N_BARS    = (R_H - HDR_H_R - 6) // BAR_ROW_H   # ~25 rows

# right panel geometry
R_LMARGIN  = 12
R_RANK_W   = 28
R_LBL_GAP  = 8
R_LABEL_W  = 185
R_BAR_GAP  = 8
R_LABEL_X  = R_LMARGIN + R_RANK_W + R_LBL_GAP          # 48
R_BAR_X    = R_LABEL_X + R_LABEL_W + R_BAR_GAP          # 241
R_PCT_W    = 62
R_RMARGIN  = 12
R_BAR_W    = R_W - R_BAR_X - R_BAR_GAP - R_PCT_W - R_RMARGIN   # ~305


def run_live_test():
    cap = cv2.VideoCapture(CAMERA_INDEX)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAMERA_WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAMERA_HEIGHT)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
    cap.set(cv2.CAP_PROP_FPS, 30)

    if not cap.isOpened():
        print('Cannot open camera!'); return

    print(f'Camera opened [{NUM_HANDS}H/{NUM_FEATURES}F].')
    print('(Q) Quit  (R/C) Clear  (SPACE) Space  (D/BACKSPACE) Delete')

    frame_buffer       = deque(maxlen=SEQUENCE_LENGTH)
    prediction_history = deque(maxlen=STABILITY_WINDOW)
    sentence_words_en  = []
    sentence_words_ar  = []
    current_word_en    = ''
    current_word_ar    = ''
    current_conf       = 0.0
    last_pred_time     = 0.0
    last_conf_time     = 0.0
    last_erase_time    = 0.0
    hand_detected      = False
    hands_count        = 0
    fps_history        = deque(maxlen=30)
    all_probs          = []   # [(en, ar, prob), ...] sorted desc

    # Colors (BGR)
    GREEN  = (0, 200, 0)
    RED    = (0, 0, 220)
    WHITE  = (255, 255, 255)
    BLACK  = (0, 0, 0)
    YELLOW = (0, 210, 210)
    ORANGE = (0, 140, 255)
    GRAY   = (70, 70, 70)
    LGRAY  = (160, 160, 160)

    while True:
        t0 = time.time()
        ret, raw = cap.read()
        if not ret:
            break

        raw = cv2.flip(raw, 1)
        now = time.time()

        # Landmark extraction on capture-size frame (upscaling first causes OOM)
        landmarks, draw_items = extract_landmarks(raw)
        hands_count  = sum(1 for t, _ in draw_items if t == 'hand')
        hand_detected = hands_count > 0
        frame_buffer.append(landmarks)

        # Prediction
        if len(frame_buffer) == SEQUENCE_LENGTH and (now - last_pred_time) >= PREDICTION_INTERVAL:
            last_pred_time = now
            seq    = np.array(frame_buffer, dtype=np.float32)
            active = np.mean(np.any(seq != 0, axis=1))
            if active >= MIN_ACTIVE_FRAMES:
                seq_batch[0] = (seq - scaler_mean) / scaler_scale
                proba        = model(seq_batch, training=False).numpy()[0]
                sorted_idx   = np.argsort(proba)[::-1]
                all_probs    = [(index_to_english.get(int(i), '?'),
                                 index_to_arabic.get(int(i),  '?'),
                                 float(proba[i])) for i in sorted_idx]
                pred_idx  = int(sorted_idx[0])
                pred_conf = float(proba[pred_idx])
                pred_en   = index_to_english.get(pred_idx, '?')
                pred_ar   = index_to_arabic.get(pred_idx,  '?')
                if pred_conf >= CONFIDENCE_THRESHOLD:
                    current_word_en = pred_en
                    current_word_ar = pred_ar
                    current_conf    = pred_conf
                    prediction_history.append(pred_en)
                    if (len(prediction_history) == STABILITY_WINDOW and
                            len(set(prediction_history)) == 1 and
                            (now - last_conf_time) >= COOLDOWN_TIME):
                        sentence_words_en.append(current_word_en)
                        sentence_words_ar.append(current_word_ar)
                        last_conf_time = now
                        prediction_history.clear()
                        print(f'Confirmed: "{current_word_ar}" ({current_conf:.1%})')
                else:
                    current_word_en = current_word_ar = ''
                    current_conf = 0.0
            else:
                current_word_en = current_word_ar = ''
                current_conf = 0.0

        # Confidence colour (shared across both panels)
        conf_clr = (GREEN  if current_conf >= 0.6 else
                    YELLOW if current_conf >= 0.4 else
                    ORANGE) if current_word_en else LGRAY

        _pil = []   # all PIL texts for combined frame (single round-trip at end)

        # ================================================================
        #  LEFT PANEL  (640 x 720) - camera + top HUD + bottom bar
        # ================================================================
        left = np.zeros((L_H, L_W, 3), dtype=np.uint8)

        # Video: landmarks drawn on resized capture frame, pasted into left panel
        vid = cv2.resize(raw, (L_W, VID_H_L))
        if DRAW_LANDMARKS:
            for item_type, lm in draw_items:
                if item_type == 'hand':
                    mp_drawing.draw_landmarks(
                        vid, lm, mp_hands.HAND_CONNECTIONS,
                        mp_drawing_styles.get_default_hand_landmarks_style(),
                        mp_drawing_styles.get_default_hand_connections_style())
                elif item_type == 'pose':
                    mp_drawing.draw_landmarks(
                        vid, lm, mp_holistic.POSE_CONNECTIONS,
                        mp_drawing_styles.get_default_pose_landmarks_style())
        left[TOP_H_L : TOP_H_L + VID_H_L] = vid

        # Erase gesture overlay
        if NUM_FEATURES == 258 and (now - last_erase_time) > 2.0:
            ls_x, ls_y = landmarks[44], landmarks[45]
            rs_x, rs_y = landmarks[48], landmarks[49]
            lw_x, lw_y = landmarks[60], landmarks[61]
            rw_x, rw_y = landmarks[64], landmarks[65]
            if ls_x and rs_x and lw_x and rw_x:
                d1 = ((lw_x-rs_x)**2 + (lw_y-rs_y)**2)**0.5
                d2 = ((rw_x-ls_x)**2 + (rw_y-ls_y)**2)**0.5
                if d1 < 0.2 and d2 < 0.2 and sentence_words_en:
                    sentence_words_en.pop()
                    if sentence_words_ar:
                        sentence_words_ar.pop()
                    cv2.putText(left, 'WORD ERASED!', (L_W//2 - 160, L_H//2),
                                FONT, 2.0, RED, 4)
                    print('Erase gesture: deleted last word.')
                    last_erase_time = now
                    frame_buffer.clear()

        # Top HUD bar
        cv2.rectangle(left, (0, 0), (L_W, TOP_H_L), BLACK, -1)
        cv2.rectangle(left, (0, 0), (L_W, TOP_H_L), WHITE, 2)

        if current_word_en:
            cv2.putText(left, 'Word:', (12, 36), FONT, 0.9, conf_clr, 2)
            _pil.append((current_word_ar or current_word_en, (120, 3), 34, conf_clr))
            stable = sum(1 for p in prediction_history if p == current_word_en)
            bx, by_b, bw_b, bh_b = 12, 50, 290, 12
            cv2.rectangle(left, (bx, by_b), (bx+bw_b, by_b+bh_b), GRAY, -1)
            cv2.rectangle(left, (bx, by_b), (bx+int(bw_b*current_conf), by_b+bh_b), conf_clr, -1)
            cv2.rectangle(left, (bx, by_b), (bx+bw_b, by_b+bh_b), WHITE, 1)
            cv2.putText(left, f'Conf: {current_conf:.1%}   Stability: {stable}/{STABILITY_WINDOW}',
                        (12, 78), FONT, 0.52, LGRAY, 1)
        else:
            status = 'Show a sign...' if hand_detected else 'No hand detected'
            cv2.putText(left, status, (12, 52), FONT, 1.0, LGRAY, 2)

        fps = 1.0 / max(time.time() - t0, 1e-6)
        fps_history.append(fps)
        avg_fps = sum(fps_history) / len(fps_history)
        cv2.putText(left, f'FPS {avg_fps:.0f}', (L_W - 90, 26), FONT, 0.55, WHITE, 1)
        cv2.putText(left, f'{NUM_HANDS}H/{NUM_FEATURES}F', (L_W-100, 50), FONT, 0.44, LGRAY, 1)

        # Status overlays above the bottom bar
        vid_bot_y = TOP_H_L + VID_H_L - 12
        buf_clr = GREEN if len(frame_buffer) >= SEQUENCE_LENGTH else YELLOW
        cv2.putText(left, f'Buf {len(frame_buffer)}/{SEQUENCE_LENGTH}',
                    (12, vid_bot_y), FONT, 0.50, buf_clr, 1)
        h_clr = ((GREEN if hands_count==2 else YELLOW if hands_count==1 else RED)
                 if NUM_HANDS==2 else (GREEN if hand_detected else RED))
        h_txt = (f'HANDS {hands_count}/2' if NUM_HANDS==2 else
                 'HAND OK' if hand_detected else 'NO HAND')
        (htw, _), _ = cv2.getTextSize(h_txt, FONT, 0.50, 1)
        cv2.putText(left, h_txt, (L_W - htw - 12, vid_bot_y), FONT, 0.50, h_clr, 1)

        # Bottom sentence bar
        by0 = TOP_H_L + VID_H_L
        cv2.rectangle(left, (0, by0), (L_W, L_H), BLACK, -1)
        cv2.rectangle(left, (0, by0), (L_W, L_H), WHITE, 2)
        sent_ar = ' '.join(sentence_words_ar) if sentence_words_ar else '(sentence will appear here)'
        sent_en = ' '.join(sentence_words_en) if sentence_words_en else ''
        cv2.putText(left, 'AR:', (12, by0+34), FONT, 0.78, (80,220,80), 2)
        _pil.append((sent_ar, (68, by0+6),  26, (80,220,80)))
        cv2.putText(left, 'EN:', (12, by0+72), FONT, 0.70, WHITE, 1)
        _pil.append((sent_en, (68, by0+50), 22, WHITE))

        # ================================================================
        #  RIGHT PANEL  (640 x 720) - all classes sorted by probability
        # ================================================================
        right = np.full((R_H, R_W, 3), (18, 18, 18), dtype=np.uint8)

        # Header
        cv2.rectangle(right, (0, 0), (R_W, HDR_H_R), (35, 35, 35), -1)
        cv2.line(right, (0, HDR_H_R), (R_W, HDR_H_R), (70, 70, 70), 1)
        n_shown = min(N_BARS, len(all_probs))
        cv2.putText(right,
                    f'All Predictions  (top {n_shown} of {len(all_probs)} classes)',
                    (12, HDR_H_R - 12), FONT, 0.46, LGRAY, 1)
        if sentence_words_ar:
            _pil.append((f'Last: {sentence_words_ar[-1]}', (L_W + R_W - 230, 5), 22, GREEN))

        # Probability bars sorted highest to lowest
        show_n = min(N_BARS, len(all_probs))
        for rank in range(show_n):
            ten, tar, tconf = all_probs[rank]
            y0 = HDR_H_R + 4 + rank * BAR_ROW_H

            bg = (26, 26, 26) if rank % 2 == 0 else (18, 18, 18)
            cv2.rectangle(right, (0, y0), (R_W, y0 + BAR_ROW_H - 1), bg, -1)

            if rank == 0:
                row_clr      = conf_clr
                bar_fill_clr = conf_clr
            elif rank < 3:
                row_clr      = (140, 220, 100)
                bar_fill_clr = (0, 160, 80)
            elif rank < 10:
                row_clr      = (90, 130, 70)
                bar_fill_clr = (0, 90, 50)
            else:
                row_clr      = (55, 65, 55)
                bar_fill_clr = (0, 50, 30)

            cv2.putText(right, f'{rank+1}.', (R_LMARGIN, y0 + 18),
                        FONT, 0.45, row_clr, 1)

            bar_fill = max(1, int(R_BAR_W * tconf))
            cv2.rectangle(right,
                          (R_BAR_X, y0+4), (R_BAR_X+R_BAR_W, y0+BAR_ROW_H-4),
                          (38, 38, 38), -1)
            cv2.rectangle(right,
                          (R_BAR_X, y0+4), (R_BAR_X+bar_fill, y0+BAR_ROW_H-4),
                          bar_fill_clr, -1)
            cv2.rectangle(right,
                          (R_BAR_X, y0+4), (R_BAR_X+R_BAR_W, y0+BAR_ROW_H-4),
                          (55, 55, 55), 1)

            pct_str = f'{tconf:.1%}'
            cv2.putText(right, pct_str,
                        (R_BAR_X + R_BAR_W + R_BAR_GAP, y0 + 18),
                        FONT, 0.43, row_clr, 1)

            label = tar if (tar and tar != ten and tar.strip() not in ('nan','?','')) else ten
            _pil.append((label, (L_W + R_LABEL_X, y0 + 4), 16, row_clr))

        # ================================================================
        #  COMBINE + DISPLAY
        # ================================================================
        combined = np.hstack([left, right])
        cv2.line(combined, (L_W, 0), (L_W, WIN_H), (80, 80, 80), 2)

        # Single PIL round-trip for ALL Arabic text across both panels
        combined = apply_pil_texts(combined, _pil)

        cv2.imshow('ArSL Word Recognition', combined)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key in (ord('r'), ord('c')):
            sentence_words_en.clear(); sentence_words_ar.clear()
            prediction_history.clear()
            current_word_en = current_word_ar = ''
            print('Cleared.')
        elif key == 32:
            sentence_words_en.append(' '); sentence_words_ar.append(' ')
        elif key in (8, 127) or key == ord('d') or key == ord('x'):
            if sentence_words_en:
                re_ = sentence_words_en.pop()
                ra_ = sentence_words_ar.pop() if sentence_words_ar else ''
                print(f'Removed: "{re_}" / "{ra_}"')

    cap.release()
    cv2.destroyAllWindows()
    if NUM_FEATURES == 258:
        detector.close()
    fe = ' '.join(sentence_words_en)
    fa = ' '.join(sentence_words_ar)
    print(f'\nFinal (EN): {fe}')
    print(f'Final (AR): {fa}')
    return fe, fa

result = run_live_test()


## Tips

| Issue | Solution |
|-------|----------|
| **Kernel crash / OOM** | Restart kernel; run Cells 1→5 only; set `DRAW_LANDMARKS = False` in Cell 2 |
| **Wrong predictions** | Model must be 48×258 — check Cell 3 prints `seq=48` |
| **Low FPS** | Use 640×480 capture (default); close other GPU apps |
| **Camera not opening** | Change `CAMERA_INDEX` to 1 |
| **Too sensitive** | Raise `STABILITY_WINDOW` to 5 or `CONFIDENCE_THRESHOLD` to 0.50 |
| **Not detecting** | Lower threshold to 0.35; hold sign ~2s for buffer to fill |

### Graduation model defaults (Cell 2)

- `arsl_custom_best.h5` + `arsl_custom_scaler.npz` + `arsl_custom_classes.csv`
- Set `USE_FILE_PICKER = True` to test older 502-class checkpoints

### How to sign

1. Face camera — upper body + both hands visible
2. Perform sign smoothly for ~2 seconds (48-frame buffer)
3. Wait for stability bar to fill → word confirmed in sentence bar